# Diabetic Retinopathy Stage Detection
**BSc Computer Science — Computer Vision Module Coursework**

This notebook implements a complete pipeline for detecting and staging Diabetic Retinopathy (DR) using
deep learning (EfficientNetB3 transfer learning) with explainability (Grad-CAM), embedding-based
similar-case retrieval, and a multi-agent clinical decision pipeline.

**Dataset:** Combined DR Dataset (APTOS + IDRiD + Messidor-2 + EyePACS subset)
**Classes (ICDR 0–4 scale):**
- 0: No DR
- 1: Mild NPDR
- 2: Moderate NPDR
- 3: Severe NPDR
- 4: Proliferative DR (PDR)

---
## Pipeline Sections
1. Setup & Data Acquisition
2. Preprocessing
3. Data Augmentation & Class Balancing
4. Train/Validation/Test Split
5. Model — CNN with Transfer Learning (EfficientNetB3)
6. Training Strategy
7. Evaluation
8. Explainability — Grad-CAM
9. Innovation A — Embedding-Based Similar-Case Retrieval
10. Innovation B — Multi-Agent Clinical Decision Pipeline
11. UI — Gradio Interface
---

## Section 1: Setup & Data Acquisition
**Goal:** Install dependencies, configure all project constants in one place (Config class),
verify GPU, download the dataset via the Kaggle API, and load/validate the labels —
including a robust fallback to folder-based labelling if no CSV is found,
corrupt-image filtering, and a class-distribution summary.

In [ ]:
# ============================================================
# SECTION 1.1 — INSTALL DEPENDENCIES
# Run this cell once. Restart the runtime if prompted after install.
# ============================================================
!pip install -q kaggle scikit-learn seaborn pillow opencv-python-headless gradio matplotlib tqdm pandas numpy tensorflow

In [ ]:
# ============================================================
# SECTION 1.2 — GLOBAL CONFIGURATION (single source of truth)
# All magic numbers and tunable hyperparameters live here.
# Modify Config values rather than hunting through code.
# ============================================================

import os
import random
import numpy as np
import tensorflow as tf


class Config:
    """
    Central configuration object.

    All project-wide constants are defined here so they can be changed in
    one place. Keeping constants here also makes experiments reproducible:
    bump SEED and everything downstream uses the new value automatically.
    """

    # --- Reproducibility ---
    SEED: int = 42

    # --- Dataset paths ---
    KAGGLE_DATASET: str = "harsha1289/combined-dr-dataset-aptosidridmessidoreyepacs"
    DATA_DIR: str       = "/content/dr_data"
    TRAIN_DIR: str      = "/content/dr_data/train"
    # CSV filenames to try, in priority order
    LABEL_CANDIDATES: list = ["train.csv", "labels.csv", "trainLabels.csv"]

    # --- Image preprocessing ---
    IMG_SIZE: int          = 224   # EfficientNetB3 default input resolution
    BEN_GRAHAM_SIGMA: int  = 10    # Gaussian blur radius for Ben Graham enhancement
    BEN_GRAHAM_ALPHA: float = 4.0  # weight on original image
    BEN_GRAHAM_BETA: float  = -4.0 # weight on blurred image (subtracted)
    BEN_GRAHAM_GAMMA: float = 128  # additive bias to centre pixel distribution

    # --- Class labels (ICDR 0-4 scale) ---
    CLASS_NAMES: list = ["No DR", "Mild", "Moderate", "Severe", "Proliferative DR"]
    NUM_CLASSES: int  = 5

    # --- Train / Val / Test split ---
    TRAIN_RATIO: float = 0.70
    VAL_RATIO: float   = 0.15
    TEST_RATIO: float  = 0.15   # three ratios must sum to 1.0

    # --- Model / Training hyperparameters ---
    BATCH_SIZE: int      = 32
    PHASE1_EPOCHS: int   = 15    # Phase 1: frozen base, train head only
    PHASE2_EPOCHS: int   = 25    # Phase 2: fine-tune top N base layers
    PHASE1_LR: float     = 1e-3  # higher LR is safe when base is frozen
    PHASE2_LR: float     = 1e-5  # very low LR prevents catastrophic forgetting
    DROPOUT_RATE: float  = 0.3   # applied before final softmax
    DENSE_UNITS: int     = 256   # units in intermediate dense layer
    UNFREEZE_TOP_N: int  = 30    # base-model layers to unfreeze in phase 2

    # --- Callbacks ---
    ES_PATIENCE: int     = 5     # early stopping patience (val_loss)
    RLROP_FACTOR: float  = 0.5   # LR reduction factor on plateau
    RLROP_PATIENCE: int  = 3     # patience before reducing LR

    # --- GovernanceAgent threshold ---
    CONFIDENCE_THRESHOLD: float = 0.70  # flag predictions below this confidence

    # --- Output paths ---
    CHECKPOINT_DIR: str  = "/content/checkpoints"
    REPORTS_DIR: str     = "/content/report_images"
    EMBEDDINGS_PATH: str = "/content/embeddings.npz"


def set_all_seeds(seed: int = Config.SEED) -> None:
    """
    Set random seeds for Python, NumPy, and TensorFlow.

    Seeds are set globally so all downstream random operations — data
    shuffling, weight initialisation, augmentation — produce the same
    results across runs, making experiments reproducible.

    Args:
        seed: Integer seed value. Defaults to Config.SEED.
    """
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    print(f"[Config] All random seeds set to {seed}.")


set_all_seeds()

In [ ]:
# ============================================================
# SECTION 1.3 — IMPORT ALL LIBRARIES
# Centralised imports make missing-dependency errors easy to spot
# and prevent the notebook from failing midway through a long run.
# ============================================================

import sys
import glob
import shutil
import warnings
import pathlib
from typing import List, Tuple, Optional, Dict

import cv2
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, cohen_kappa_score

from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

warnings.filterwarnings("ignore")  # suppress non-critical deprecation warnings

# Create all output directories upfront — avoids failures in later sections
for _dir in [Config.CHECKPOINT_DIR, Config.REPORTS_DIR, Config.DATA_DIR]:
    os.makedirs(_dir, exist_ok=True)

print(f"Python     : {sys.version}")
print(f"TensorFlow : {tf.__version__}")
print(f"Keras      : {keras.__version__}")
print("All libraries imported successfully.")

In [ ]:
# ============================================================
# SECTION 1.4 — GPU VERIFICATION
# Training on CPU for a 21k-image dataset takes many hours.
# We detect GPUs here and warn loudly if none is found.
# In Colab: Runtime > Change runtime type > GPU (T4 or A100).
# ============================================================

def verify_gpu() -> None:
    """
    Detect and report available GPU devices.

    Enables memory growth on each GPU to prevent TensorFlow from
    grabbing all VRAM at startup, which can cause OOM errors when
    sharing a Colab GPU with other processes.

    Prints a warning (but does not raise) if no GPU is found, so
    the notebook can still be used for small-batch testing on CPU.
    """
    gpus = tf.config.list_physical_devices("GPU")
    if gpus:
        for gpu in gpus:
            # Incremental VRAM allocation rather than reserving all memory up-front
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"[GPU] {len(gpus)} GPU(s) detected:")
        for g in gpus:
            print(f"      {g.name}")
        os.system("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null")
    else:
        print(
            "[WARNING] No GPU detected. Training will be very slow on CPU.\n"
            "          In Google Colab: Runtime > Change runtime type > GPU."
        )


verify_gpu()

In [ ]:
# ============================================================
# SECTION 1.5 — KAGGLE API DOWNLOAD
# The Kaggle API requires a personal API token (kaggle.json).
# In Colab this cell prompts an upload dialog.
# Locally it reads kaggle.json from the current working directory.
# ============================================================

def setup_kaggle_credentials() -> None:
    """
    Place kaggle.json in ~/.kaggle/kaggle.json with correct permissions.

    The Kaggle CLI refuses to run if permissions on kaggle.json are too
    permissive (it is a security token). We enforce 600 (owner read/write).

    Raises:
        FileNotFoundError: If kaggle.json cannot be found or uploaded.
    """
    kaggle_dir = os.path.expanduser("~/.kaggle")
    os.makedirs(kaggle_dir, exist_ok=True)
    target = os.path.join(kaggle_dir, "kaggle.json")

    if os.path.exists(target):
        print("[Kaggle] kaggle.json already in place — skipping upload.")
        return

    try:
        from google.colab import files  # type: ignore
        print("[Kaggle] Please upload your kaggle.json when the dialog appears...")
        uploaded = files.upload()
        if "kaggle.json" not in uploaded:
            raise FileNotFoundError(
                "kaggle.json was not found in the uploaded files. "
                "Download it from https://www.kaggle.com/settings > API > Create New Token."
            )
        shutil.move("kaggle.json", target)
    except ImportError:
        # Not in Colab — look for kaggle.json in the current directory
        if os.path.exists("kaggle.json"):
            shutil.copy("kaggle.json", target)
        else:
            raise FileNotFoundError(
                "kaggle.json not found in the current directory. "
                "Place it here or run in Colab for an upload prompt."
            )

    # Restrict permissions — required by the Kaggle CLI security check
    os.chmod(target, 0o600)
    print(f"[Kaggle] Credentials saved to {target} with 600 permissions.")


def download_dataset(force_redownload: bool = False) -> None:
    """
    Download and extract the Combined DR Dataset from Kaggle.

    Idempotent: skips the download if images are already present on disk.
    Set force_redownload=True to re-fetch the dataset regardless.

    Args:
        force_redownload: Re-download even if data already exists on disk.

    Raises:
        RuntimeError: If the Kaggle CLI returns a non-zero exit code.
    """
    existing = (
        glob.glob(os.path.join(Config.DATA_DIR, "**", "*.jpeg"), recursive=True)
        + glob.glob(os.path.join(Config.DATA_DIR, "**", "*.jpg"),  recursive=True)
        + glob.glob(os.path.join(Config.DATA_DIR, "**", "*.png"),  recursive=True)
    )
    if existing and not force_redownload:
        print(
            f"[Dataset] {len(existing):,} images already in {Config.DATA_DIR}. "
            "Skipping download. Pass force_redownload=True to override."
        )
        return

    print(f"[Dataset] Downloading '{Config.KAGGLE_DATASET}' — this may take several minutes...")
    # --unzip extracts automatically; -q suppresses verbose progress output
    exit_code = os.system(
        f"kaggle datasets download -d {Config.KAGGLE_DATASET} "
        f"-p {Config.DATA_DIR} --unzip -q"
    )
    if exit_code != 0:
        raise RuntimeError(
            f"Kaggle download failed (exit code {exit_code}). "
            "Check your kaggle.json credentials and the dataset slug."
        )
    print(f"[Dataset] Download complete -> {Config.DATA_DIR}")


setup_kaggle_credentials()
download_dataset()

In [ ]:
# ============================================================
# SECTION 1.6 — DISCOVER DATASET STRUCTURE
# Print the top-level directory tree so we know exactly what Kaggle
# delivered before attempting label loading.
# ============================================================

def print_directory_tree(root: str, max_depth: int = 3, max_items: int = 20) -> None:
    """
    Print a human-readable directory tree for quick structure inspection.

    Args:
        root:      Root directory to start from.
        max_depth: Maximum folder depth to recurse into.
        max_items: Max items shown per directory (prevents flooding output).
    """
    root_path = pathlib.Path(root)
    if not root_path.exists():
        print(f"[Tree] Path does not exist: {root}")
        return

    def _recurse(path: pathlib.Path, depth: int, prefix: str) -> None:
        if depth > max_depth:
            return
        try:
            children = sorted(path.iterdir())
        except PermissionError:
            return
        dirs  = [c for c in children if c.is_dir()]
        files = [c for c in children if c.is_file()]
        items = dirs + files
        truncated = len(items) > max_items
        items = items[:max_items]
        for i, item in enumerate(items):
            is_last = (i == len(items) - 1) and not truncated
            connector = "\u2514\u2500\u2500 " if is_last else "\u251c\u2500\u2500 "
            suffix = "/" if item.is_dir() else ""
            print(f"{prefix}{connector}{item.name}{suffix}")
            if item.is_dir():
                ext = "    " if is_last else "\u2502   "
                _recurse(item, depth + 1, prefix + ext)
        if truncated:
            print(f"{prefix}    ... (showing first {max_items} items)")

    print(f"\n[Tree] {root}/")
    _recurse(root_path, 1, "")


print_directory_tree(Config.DATA_DIR)

In [ ]:
# ============================================================
# SECTION 1.7 — LABEL LOADING (CSV-first, folder fallback)
#
# Loading strategy:
#   1. Search for a recognised CSV file (Config.LABEL_CANDIDATES).
#   2. If found: parse and normalise all label values to ICDR 0-4.
#   3. If not found: derive labels from subfolder names.
#
# Messidor-2 note:
#   Messidor-2 originally used a 4-level scale (grades 0-3) where
#   grade 3 covers both Severe and Proliferative DR. The Kaggle
#   combined dataset re-maps everything to ICDR 0-4 before upload, so
#   inconsistencies should be rare — but we validate and flag any that
#   slip through rather than silently accepting them.
# ============================================================

# Column name candidates across sub-datasets
_LABEL_COL_CANDIDATES = ["diagnosis", "label", "level", "dr_grade", "grade", "class"]
_IMG_COL_CANDIDATES   = ["id_code", "image", "filename", "image_name", "id"]

# Folder name -> ICDR integer mapping (covers common naming conventions)
_FOLDER_LABEL_MAP = {
    "0": 0, "no_dr": 0, "nodr": 0, "normal": 0,
    "1": 1, "mild": 1,
    "2": 2, "moderate": 2,
    "3": 3, "severe": 3,
    "4": 4, "proliferative": 4, "pdr": 4,
}


def _find_csv(search_root: str) -> Optional[str]:
    """
    Search for a label CSV file inside search_root.

    Tries Config.LABEL_CANDIDATES at the root level first, then falls
    back to a recursive glob one level deeper.

    Args:
        search_root: Directory to search.

    Returns:
        Absolute path to the first matching CSV, or None if not found.
    """
    for candidate in Config.LABEL_CANDIDATES:
        fp = os.path.join(search_root, candidate)
        if os.path.exists(fp):
            return fp
    for candidate in Config.LABEL_CANDIDATES:
        matches = glob.glob(os.path.join(search_root, "**", candidate), recursive=True)
        if matches:
            return matches[0]
    return None


def _normalise_csv_labels(df: pd.DataFrame) -> pd.DataFrame:
    """
    Detect and standardise label and image-path columns in a raw CSV DataFrame.

    Handles differing column names (APTOS uses 'diagnosis', EyePACS uses 'level',
    IDRiD uses 'DR_grade') and string labels ("Mild", "moderate") across
    sub-datasets. Maps all values to ICDR integers 0-4.

    Args:
        df: Raw DataFrame loaded from pd.read_csv().

    Returns:
        DataFrame with exactly two columns: 'filepath' (str) and 'label' (int 0-4).

    Raises:
        ValueError: If no recognisable label or image-filename column is found.
    """
    df.columns = [c.strip().lower() for c in df.columns]

    label_col = next((c for c in _LABEL_COL_CANDIDATES if c in df.columns), None)
    if label_col is None:
        raise ValueError(
            f"No label column found. Columns present: {list(df.columns)}. "
            f"Expected one of: {_LABEL_COL_CANDIDATES}"
        )

    img_col = next((c for c in _IMG_COL_CANDIDATES if c in df.columns), None)
    if img_col is None:
        raise ValueError(
            f"No image-filename column found. Columns present: {list(df.columns)}. "
            f"Expected one of: {_IMG_COL_CANDIDATES}"
        )

    # Map string labels to ICDR integers
    string_map = {
        "no dr": 0, "no_dr": 0, "nodr": 0, "normal": 0, "0": 0,
        "mild": 1, "mild npdr": 1, "1": 1,
        "moderate": 2, "moderate npdr": 2, "2": 2,
        "severe": 3, "severe npdr": 3, "3": 3,
        "proliferative dr": 4, "pdr": 4, "proliferative": 4, "4": 4,
    }
    raw = df[label_col].astype(str).str.strip().str.lower()
    df["label"] = raw.map(string_map)

    # For any still-NaN labels, try direct numeric conversion
    nan_mask = df["label"].isna()
    if nan_mask.any():
        df.loc[nan_mask, "label"] = pd.to_numeric(
            df.loc[nan_mask, label_col], errors="coerce"
        )

    df["filepath"] = df[img_col].astype(str)
    return df[["filepath", "label"]]


def _load_from_folders(image_root: str) -> pd.DataFrame:
    """
    Build a label DataFrame from a folder-structured dataset.

    Expected structure::

        image_root/
            0/  (or No_DR/)
                image001.jpeg
            1/  (or Mild/)
                ...

    Args:
        image_root: Root directory with one subdirectory per class.

    Returns:
        DataFrame with 'filepath' (absolute) and 'label' (int 0-4).

    Raises:
        FileNotFoundError: If image_root does not exist or contains no images.
    """
    if not os.path.isdir(image_root):
        raise FileNotFoundError(f"Image root directory not found: {image_root}")

    records = []
    valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tiff"}

    for folder_name in sorted(os.listdir(image_root)):
        folder_path = os.path.join(image_root, folder_name)
        if not os.path.isdir(folder_path):
            continue
        key = folder_name.strip().lower().replace(" ", "_")
        label_int = _FOLDER_LABEL_MAP.get(key)
        if label_int is None:
            print(
                f"[Label] WARNING: folder '{folder_name}' cannot be mapped to an ICDR class "
                "and will be skipped. Add it to _FOLDER_LABEL_MAP if needed."
            )
            continue
        for fname in os.listdir(folder_path):
            if pathlib.Path(fname).suffix.lower() in valid_exts:
                records.append({
                    "filepath": os.path.join(folder_path, fname),
                    "label": label_int,
                })

    if not records:
        raise FileNotFoundError(
            f"No images found under {image_root} via folder-based labelling. "
            "Ensure subdirectory names match ICDR class names (0-4, mild, etc.)."
        )

    print(f"[Label] Folder-based labelling: {len(records):,} images found.")
    return pd.DataFrame(records)


def load_labels(data_dir: str, image_root: str) -> pd.DataFrame:
    """
    Load dataset labels using a CSV-first, folder-fallback strategy.

    After loading, validates that all labels are in {0,1,2,3,4}, drops
    rows with unmappable values, resolves relative paths to absolute paths,
    and filters out rows where the image file does not exist on disk.

    Args:
        data_dir:   Root dataset directory (searched for CSVs).
        image_root: Directory containing the actual image files.

    Returns:
        Cleaned DataFrame with columns 'filepath' (str) and 'label' (int 0-4).
    """
    csv_path = _find_csv(data_dir)

    if csv_path:
        print(f"[Label] CSV found: {csv_path}")
        raw_df = pd.read_csv(csv_path)
        print(f"[Label] {len(raw_df):,} rows loaded from CSV.")
        df = _normalise_csv_labels(raw_df)

        # Resolve relative CSV paths to absolute filesystem paths
        def _resolve(fp: str) -> str:
            if os.path.isabs(fp) and os.path.exists(fp):
                return fp
            candidate = os.path.join(image_root, fp)
            if os.path.exists(candidate):
                return candidate
            # Try adding common image extensions (some CSVs omit the extension)
            base = os.path.splitext(fp)[0]
            for ext in [".jpeg", ".jpg", ".png"]:
                c = os.path.join(image_root, base + ext)
                if os.path.exists(c):
                    return c
            return fp  # will be caught by missing-file filter below

        df["filepath"] = df["filepath"].apply(_resolve)
    else:
        print(f"[Label] No CSV found in {data_dir}. Falling back to folder-based labelling.")
        df = _load_from_folders(image_root)

    # Validate label values
    df["label"] = pd.to_numeric(df["label"], errors="coerce")
    bad_mask = ~df["label"].isin({0, 1, 2, 3, 4}) | df["label"].isna()
    if bad_mask.any():
        bad_vals = df.loc[bad_mask, "label"].unique().tolist()[:10]
        print(
            f"[Label] WARNING: {bad_mask.sum()} rows with out-of-range labels will be DROPPED. "
            f"Unmappable values: {bad_vals}\n"
            "        This may indicate Messidor-2 grading inconsistencies or annotation errors."
        )
        df = df[~bad_mask].copy()

    df["label"] = df["label"].astype(int)

    # Filter rows where the image file does not exist on disk
    exists_mask = df["filepath"].apply(os.path.exists)
    n_missing = (~exists_mask).sum()
    if n_missing > 0:
        print(f"[Label] WARNING: {n_missing} image(s) referenced in labels not found on disk. Removing.")
    df = df[exists_mask].reset_index(drop=True)
    print(f"[Label] Final usable dataset size: {len(df):,} images.")
    return df


df_labels = load_labels(Config.DATA_DIR, Config.TRAIN_DIR)
df_labels.head()

In [ ]:
# ============================================================
# SECTION 1.8 — CORRUPT IMAGE DETECTION
# Corrupt images (truncated, zero-byte, unreadable) crash the pipeline
# or silently produce garbage. PIL.Image.verify() reads only the header
# and raises on corrupt files — much faster than full pixel decoding.
# ============================================================

def filter_corrupt_images(
    df: pd.DataFrame, filepath_col: str = "filepath"
) -> pd.DataFrame:
    """
    Remove rows referencing corrupt or unreadable image files.

    Uses PIL.Image.verify() which reads image metadata without decoding
    pixel data. This catches JPEG truncation and file-format errors that
    OpenCV silently ignores (returning a black or partial image instead).

    Args:
        df:           DataFrame with a file path column.
        filepath_col: Name of the file path column.

    Returns:
        DataFrame with corrupt-image rows removed.
    """
    corrupt_indices = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Checking image integrity"):
        try:
            with Image.open(row[filepath_col]) as img:
                img.verify()  # header-only check — does not decode pixel data
        except (UnidentifiedImageError, OSError, SyntaxError):
            # SyntaxError is raised by PIL for some truncated JPEG files
            corrupt_indices.append(idx)

    if corrupt_indices:
        examples = df.loc[corrupt_indices, filepath_col].tolist()[:5]
        print(f"[Corrupt] {len(corrupt_indices)} corrupt image(s) found and removed. Examples:")
        for p in examples:
            print(f"  {p}")
        df = df.drop(index=corrupt_indices).reset_index(drop=True)
    else:
        print("[Corrupt] All images passed integrity check — dataset is clean.")

    return df


df_labels = filter_corrupt_images(df_labels)
print(f"Dataset size after integrity check: {len(df_labels):,} images.")

In [ ]:
# ============================================================
# SECTION 1.9 — CLASS DISTRIBUTION SUMMARY & VISUALISATION
# Print a per-class count table and plot a bar chart.
# The expected severe imbalance (No DR ~14,063 vs Proliferative ~702,
# roughly 20x) is documented here. This is the primary motivation for
# applying sklearn compute_class_weight in Section 3.
# ============================================================

def print_class_distribution(df: pd.DataFrame, label_col: str = "label") -> pd.DataFrame:
    """
    Compute and print a class distribution summary table.

    Args:
        df:        DataFrame with an integer label column.
        label_col: Name of the label column.

    Returns:
        Summary DataFrame with columns: class_id, class_name, count, percentage.
    """
    counts = df[label_col].value_counts().sort_index()
    total  = counts.sum()

    summary = pd.DataFrame({
        "class_id":   counts.index,
        "class_name": [Config.CLASS_NAMES[i] for i in counts.index],
        "count":      counts.values,
        "percentage": (counts.values / total * 100).round(2),
    })

    print("\n" + "=" * 58)
    print(f"  CLASS DISTRIBUTION  (total: {total:,} images)")
    print("=" * 58)
    print(summary.to_string(index=False))
    print("-" * 58)
    ratio = counts.max() / counts.min()
    print(f"  Imbalance ratio (max / min class): {ratio:.1f}x")
    if ratio > 5:
        print(
            f"  [ACTION] Severe imbalance detected ({ratio:.1f}x). "
            "class_weight will be applied during training in Section 6."
        )
    print("=" * 58 + "\n")
    return summary


def plot_class_distribution(
    df: pd.DataFrame,
    label_col: str = "label",
    save_path: Optional[str] = None,
) -> None:
    """
    Plot and optionally save a class count bar chart.

    Args:
        df:        DataFrame with label column.
        label_col: Integer label column name.
        save_path: If provided, saves the figure as a PNG at this path.
    """
    counts = df[label_col].value_counts().sort_index()

    fig, ax = plt.subplots(figsize=(9, 5))
    palette = sns.color_palette("Blues_d", len(Config.CLASS_NAMES))
    bars = ax.bar(
        Config.CLASS_NAMES,
        counts.values,
        color=palette,
        edgecolor="black",
        linewidth=0.7,
    )
    for bar, count in zip(bars, counts.values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 60,
            f"{count:,}",
            ha="center", va="bottom", fontsize=10, fontweight="bold",
        )

    ax.set_title("Class Distribution — Combined DR Dataset", fontsize=14, pad=15)
    ax.set_xlabel("DR Stage (ICDR 0-4 Scale)", fontsize=12)
    ax.set_ylabel("Number of Images", fontsize=12)
    ax.set_ylim(0, counts.max() * 1.15)
    plt.tight_layout()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"[Plot] Saved -> {save_path}")

    plt.show()


class_summary = print_class_distribution(df_labels)
plot_class_distribution(
    df_labels,
    save_path=os.path.join(Config.REPORTS_DIR, "class_distribution.png"),
)

In [ ]:
# ============================================================
# SECTION 1.10 — IMAGE DIMENSION AUDIT
# Fundus photographs vary enormously in native resolution across
# sub-datasets (APTOS: ~2048x2048, IDRiD: ~4288x2848, Messidor: varies).
# We sample 500 images to report the range so the preprocessing section
# is designed with accurate expectations.
# ============================================================

def audit_image_dimensions(
    df: pd.DataFrame,
    sample_n: int = 500,
    filepath_col: str = "filepath",
) -> None:
    """
    Sample images and report width/height statistics.

    Reads only the image header (not full pixel data) via PIL, so
    auditing 500 images takes only a few seconds regardless of resolution.

    Args:
        df:           DataFrame with filepath column.
        sample_n:     Number of images to randomly sample.
        filepath_col: Name of the file path column.
    """
    sample = df.sample(min(sample_n, len(df)), random_state=Config.SEED)
    widths, heights = [], []

    for fp in tqdm(sample[filepath_col], desc="Auditing dimensions"):
        try:
            with Image.open(fp) as img:
                w, h = img.size  # PIL: (width, height)
                widths.append(w)
                heights.append(h)
        except Exception:
            pass  # already screened by corrupt filter

    w = np.array(widths)
    h = np.array(heights)
    print(f"\n[Dimensions] Sampled {len(w)} images:")
    print(f"  Width  — min:{w.min():6d}  max:{w.max():6d}  mean:{w.mean():.0f}")
    print(f"  Height — min:{h.min():6d}  max:{h.max():6d}  mean:{h.mean():.0f}")
    print(f"  All images will be resized to {Config.IMG_SIZE}x{Config.IMG_SIZE} in Section 2.")


audit_image_dimensions(df_labels)

# ---- Section 1 complete ----
print("\n" + "=" * 60)
print("  SECTION 1 COMPLETE")
print(f"  Total images loaded and validated : {len(df_labels):,}")
print(f"  Label column                      : 'label'    (int, ICDR 0-4)")
print(f"  Image path column                 : 'filepath' (absolute path)")
print("  Next -> Section 2: Preprocessing")
print("=" * 60)

## Section 2: Preprocessing

**Goal:** Transform raw fundus photographs into clean, normalised 224×224 tensors
ready for the model. Steps applied to every image, in order:

1. **Black-border crop** — retinal fundus images are circular; the surrounding
   black padding contains no information and wastes model capacity.
2. **Resize** to 224×224 (EfficientNetB3 input size).
3. **Ben Graham contrast enhancement** — a specific named technique that
   suppresses low-frequency illumination variation and enhances local vessel
   contrast. Implemented explicitly (not replaced with CLAHE).
4. **Normalise** pixel values to [0, 1].

Before/after comparison images are saved to `Config.REPORTS_DIR` for the report.

In [ ]:
# ============================================================
# SECTION 2.1 — IMAGE LOADING UTILITY
# A single function that loads an image file into an RGB NumPy
# array, used as the entry point for every preprocessing step.
# ============================================================

def load_image_rgb(filepath: str) -> np.ndarray:
    """
    Load an image from disk and convert to an RGB NumPy array.

    Uses OpenCV (fast, handles many formats) and converts BGR -> RGB
    so colours are correct for matplotlib and TensorFlow.

    Args:
        filepath: Absolute path to the image file.

    Returns:
        NumPy array of shape (H, W, 3), dtype uint8, RGB channel order.

    Raises:
        FileNotFoundError: If the file does not exist.
        ValueError:        If OpenCV fails to decode the image.
    """
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Image not found: {filepath}")

    img_bgr = cv2.imread(filepath)
    if img_bgr is None:
        raise ValueError(
            f"OpenCV could not decode image: {filepath}. "
            "The file may be corrupt or in an unsupported format."
        )
    # OpenCV loads as BGR; convert to RGB for compatibility with PIL and TF
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

In [ ]:
# ============================================================
# SECTION 2.2 — BLACK-BORDER CROPPING
#
# WHY: Fundus cameras capture a circular retinal image centred
# on a rectangular sensor. The corners of the photograph are
# pure black (pixel value ≈ 0). This padding:
#   - Wastes model capacity on uninformative pixels
#   - Can confuse augmentation (random crops may include mostly black)
#   - Inflates the apparent image size
#
# APPROACH (Ben Graham / crop_image_from_gray style):
#   1. Convert to grayscale.
#   2. Apply a binary threshold at a low value (e.g. 7) to create a
#      mask that is 1 where the retina is and 0 where it is black.
#   3. Find the bounding box of the non-zero mask region.
#   4. Crop the original colour image to that bounding box.
#   5. If the crop is degenerate (all black image), return the original.
# ============================================================

def crop_image_from_gray(
    img: np.ndarray,
    threshold: int = 7,
    tol: int = 7,
) -> np.ndarray:
    """
    Remove black borders from a fundus photograph.

    The retinal fundus image is circular; surrounding pixels are near-black
    (value < threshold). We threshold the grayscale image to find the retinal
    disc boundary and crop the colour image to that bounding box.

    This is commonly called the "crop_image_from_gray" technique, popularised
    by Ben Graham's APTOS competition solution.

    Args:
        img:       RGB NumPy array (H, W, 3), uint8.
        threshold: Grayscale pixel value below which a pixel is considered
                   background (black border). Default 7 is empirically robust
                   across the APTOS, IDRiD, and Messidor datasets.
        tol:       Additional tolerance in pixels to expand the crop slightly
                   and avoid clipping the very edge of the optic disc.

    Returns:
        Cropped RGB NumPy array. Falls back to the original if the crop is
        degenerate (e.g. an entirely black image).
    """
    if img.ndim == 2:
        # Already grayscale — work directly
        mask = img > threshold
        if not mask.any():
            return img  # guard: entirely black image
        row_mask = mask.any(axis=1)
        col_mask = mask.any(axis=0)
        rmin, rmax = np.where(row_mask)[0][[0, -1]]
        cmin, cmax = np.where(col_mask)[0][[0, -1]]
        # Add tolerance so we do not clip the disc edge
        rmin = max(0, rmin - tol)
        rmax = min(img.shape[0] - 1, rmax + tol)
        cmin = max(0, cmin - tol)
        cmax = min(img.shape[1] - 1, cmax + tol)
        return img[rmin:rmax + 1, cmin:cmax + 1]

    # For an RGB image, compute the grayscale mask from a single channel
    # (green channel has best contrast for retinal vessels)
    gray = img[:, :, 1]
    mask = gray > threshold

    if not mask.any():
        return img  # guard: entirely black or near-black image

    row_mask = mask.any(axis=1)
    col_mask = mask.any(axis=0)
    rmin, rmax = np.where(row_mask)[0][[0, -1]]
    cmin, cmax = np.where(col_mask)[0][[0, -1]]

    rmin = max(0, rmin - tol)
    rmax = min(img.shape[0] - 1, rmax + tol)
    cmin = max(0, cmin - tol)
    cmax = min(img.shape[1] - 1, cmax + tol)

    cropped = img[rmin:rmax + 1, cmin:cmax + 1]

    # Sanity check: if the crop is nearly empty, return the original
    if cropped.size == 0 or min(cropped.shape[:2]) < 10:
        return img

    return cropped

In [ ]:
# ============================================================
# SECTION 2.3 — BEN GRAHAM CONTRAST ENHANCEMENT
#
# WHY: Fundus photographs suffer from uneven illumination — the
# centre of the image is often brighter than the periphery due to
# the ophthalmoscope light path. This hides subtle lesion contrast
# (microaneurysms, haemorrhages, exudates) in darker regions.
#
# THE TECHNIQUE: Subtract a heavily blurred (low-frequency) version
# of the image from the original, then re-centre the distribution:
#
#   enhanced = alpha * original + beta * GaussianBlur(original, sigma) + gamma
#
# With alpha=4, beta=-4, gamma=128:
#   - The blurred image captures the slow illumination gradient
#   - Subtracting it removes that gradient (local contrast normalisation)
#   - gamma=128 re-centres pixel values near the middle of [0,255]
#     so we do not clip large negative or positive residuals
#   - alpha=4 amplifies fine-detail (vessel edges, lesion boundaries)
#
# This is NOT the same as CLAHE (which operates on histograms of tiles);
# Ben Graham enhancement operates globally in the spatial domain and
# specifically targets the illumination gradient rather than histogram spread.
# It was used by the winning solution of the 2015 Kaggle DR competition.
# ============================================================

def ben_graham_enhance(
    img: np.ndarray,
    sigma: int = Config.BEN_GRAHAM_SIGMA,
    alpha: float = Config.BEN_GRAHAM_ALPHA,
    beta: float  = Config.BEN_GRAHAM_BETA,
    gamma: float = Config.BEN_GRAHAM_GAMMA,
) -> np.ndarray:
    """
    Apply Ben Graham-style contrast enhancement to a fundus image.

    Subtracts a Gaussian-blurred version of the image from itself to
    remove the slow illumination gradient present in fundus photography,
    then re-centres pixel values with an additive bias.

    Formula: enhanced = clip(alpha * img + beta * blur(img, sigma) + gamma)

    Args:
        img:   RGB uint8 NumPy array (H, W, 3).
        sigma: Gaussian blur kernel sigma. Large sigma (default 10) captures
               the broad illumination gradient across the whole image.
        alpha: Weight on the original image. Value of 4 amplifies fine detail.
        beta:  Weight on the blurred image (typically -alpha to subtract it).
        gamma: Additive constant to re-centre the output around 128.

    Returns:
        Enhanced RGB uint8 NumPy array, same shape as input.
    """
    # Build the blur kernel size from sigma: must be odd and positive
    # ksize = 2 * (4 * sigma) + 1 is the standard "full-width" Gaussian kernel
    ksize = int(2 * round(4 * sigma) + 1)

    # Gaussian blur captures the low-frequency illumination pattern
    blurred = cv2.GaussianBlur(img, (ksize, ksize), sigma)

    # Weighted addition: original (amplified) minus blur (background), re-centred
    enhanced = cv2.addWeighted(img, alpha, blurred, beta, gamma)

    # clip is implicit in addWeighted for uint8, but we apply it explicitly
    # to avoid artefacts if the input dtype differs
    enhanced = np.clip(enhanced, 0, 255).astype(np.uint8)
    return enhanced

In [ ]:
# ============================================================
# SECTION 2.4 — FULL PREPROCESSING PIPELINE
# Chains the three steps (crop -> resize -> Ben Graham -> normalise)
# into one function that returns a float32 tensor ready for the model.
# ============================================================

def preprocess_image(
    filepath: str,
    img_size: int = Config.IMG_SIZE,
    apply_ben_graham: bool = True,
) -> np.ndarray:
    """
    Full preprocessing pipeline for a single fundus image.

    Steps applied in order:
    1. Load image as RGB uint8.
    2. Crop black borders (crop_image_from_gray).
    3. Resize to (img_size, img_size) using bilinear interpolation.
    4. Apply Ben Graham contrast enhancement (if apply_ben_graham=True).
    5. Normalise pixel values to [0.0, 1.0] (float32).

    Args:
        filepath:         Absolute path to the image file.
        img_size:         Target side length in pixels (square output).
        apply_ben_graham: Whether to apply Ben Graham enhancement.
                          Set False for ablation studies.

    Returns:
        NumPy float32 array of shape (img_size, img_size, 3), values in [0, 1].

    Raises:
        FileNotFoundError: Propagated from load_image_rgb if file is missing.
        ValueError:        Propagated from load_image_rgb if file is corrupt.
    """
    # Step 1: Load
    img = load_image_rgb(filepath)

    # Step 2: Crop black borders
    # Done before resizing so we do not waste model capacity on padding
    img = crop_image_from_gray(img)

    # Step 3: Resize to the network input size
    # INTER_AREA is best for downscaling (anti-aliasing); INTER_LINEAR for upscaling
    h, w = img.shape[:2]
    interp = cv2.INTER_AREA if (h > img_size or w > img_size) else cv2.INTER_LINEAR
    img = cv2.resize(img, (img_size, img_size), interpolation=interp)

    # Step 4: Ben Graham contrast enhancement (applied after resize so the
    # Gaussian kernel size is consistent across all images regardless of
    # their original resolution)
    if apply_ben_graham:
        img = ben_graham_enhance(img)

    # Step 5: Normalise to [0, 1] — required by EfficientNetB3 when using
    # the default Keras preprocessing (which expects 0-1 float32 input)
    img = img.astype(np.float32) / 255.0

    return img


# Quick sanity check on the first image in the dataset
_test_fp = df_labels["filepath"].iloc[0]
_test_out = preprocess_image(_test_fp)
print(f"Preprocessing check:")
print(f"  Input file  : {_test_fp}")
print(f"  Output shape: {_test_out.shape}")
print(f"  Dtype       : {_test_out.dtype}")
print(f"  Value range : [{_test_out.min():.3f}, {_test_out.max():.3f}]")

In [ ]:
# ============================================================
# SECTION 2.5 — BEFORE / AFTER COMPARISON VISUALISATION
# Saves side-by-side comparison grids for the report.
# We show: (a) original raw image, (b) after crop+resize,
# (c) after Ben Graham enhancement. One grid per DR class.
# ============================================================

def visualise_preprocessing(
    df: pd.DataFrame,
    n_per_class: int = 2,
    save_dir: str = Config.REPORTS_DIR,
) -> None:
    """
    Generate and save before/after preprocessing comparison figures.

    For each DR class, randomly samples n_per_class images and renders
    a 3-column grid showing: raw, cropped+resized, and fully enhanced.
    Figures are saved to save_dir for inclusion in the report.

    Args:
        df:           DataFrame with 'filepath' and 'label' columns.
        n_per_class:  Number of example images per class to display.
        save_dir:     Directory where PNG comparison files are saved.
    """
    os.makedirs(save_dir, exist_ok=True)

    for class_id, class_name in enumerate(Config.CLASS_NAMES):
        class_df = df[df["label"] == class_id]
        if class_df.empty:
            print(f"[Preprocess] WARNING: No images found for class {class_id} ({class_name}). Skipping.")
            continue

        # Sample up to n_per_class images (fewer if the class is small)
        sample = class_df.sample(min(n_per_class, len(class_df)), random_state=Config.SEED)

        n_cols = 3   # raw | cropped+resized | Ben Graham enhanced
        n_rows = len(sample)
        fig, axes = plt.subplots(
            n_rows, n_cols,
            figsize=(n_cols * 4, n_rows * 4),
        )
        # Ensure axes is always 2D even for n_rows=1
        if n_rows == 1:
            axes = axes[np.newaxis, :]

        col_titles = ["Original (raw)", "Cropped + Resized", "Ben Graham Enhanced"]
        for ax, title in zip(axes[0], col_titles):
            ax.set_title(title, fontsize=10, fontweight="bold")

        for row_idx, (_, row) in enumerate(sample.iterrows()):
            fp = row["filepath"]

            # Column 0: original raw image
            raw = load_image_rgb(fp)
            axes[row_idx, 0].imshow(raw)
            axes[row_idx, 0].axis("off")

            # Column 1: cropped and resized (no Ben Graham)
            cropped = crop_image_from_gray(raw)
            resized = cv2.resize(
                cropped,
                (Config.IMG_SIZE, Config.IMG_SIZE),
                interpolation=cv2.INTER_AREA if (
                    cropped.shape[0] > Config.IMG_SIZE
                ) else cv2.INTER_LINEAR,
            )
            axes[row_idx, 1].imshow(resized)
            axes[row_idx, 1].axis("off")

            # Column 2: fully processed (Ben Graham applied to the resized image)
            enhanced = ben_graham_enhance(resized)
            axes[row_idx, 2].imshow(enhanced)
            axes[row_idx, 2].axis("off")

        fig.suptitle(
            f"Preprocessing Steps — Class {class_id}: {class_name}",
            fontsize=13,
            fontweight="bold",
            y=1.01,
        )
        plt.tight_layout()

        save_path = os.path.join(save_dir, f"preprocessing_class{class_id}_{class_name.replace(' ', '_')}.png")
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"[Preprocess] Saved comparison figure -> {save_path}")


visualise_preprocessing(df_labels, n_per_class=2)

In [ ]:
# ============================================================
# SECTION 2.6 — TF.DATA PIPELINE
#
# WHY tf.data: Loading and preprocessing images on-the-fly in
# Python (with a plain Python generator) creates a CPU bottleneck
# that starves the GPU. tf.data pipelines:
#   - Use parallel map workers (num_parallel_calls) so multiple
#     images are preprocessed concurrently on the CPU
#   - Use prefetch so the next batch is ready before the GPU
#     finishes the current one — zero GPU idle time
#   - Cache decoded images to RAM on second-epoch replay if the
#     dataset fits in memory (disabled here for safety with 21k images)
#
# The pipeline wraps preprocess_image in a tf.py_function so our
# existing NumPy/OpenCV preprocessing functions work seamlessly
# inside the TF graph.
# ============================================================

def _preprocess_tf(filepath: tf.Tensor, label: tf.Tensor) -> tuple:
    """
    TensorFlow-compatible wrapper around preprocess_image.

    Called by tf.data.Dataset.map(). Uses tf.py_function to run
    our NumPy/OpenCV preprocessing inside the TF graph.

    Args:
        filepath: Scalar string tensor containing the image file path.
        label:    Scalar int32 tensor containing the ICDR label.

    Returns:
        Tuple of:
            img   — float32 tensor (IMG_SIZE, IMG_SIZE, 3)
            label — int32 tensor (scalar)
    """
    def _py_preprocess(fp_bytes):
        # Decode the byte string tensor to a Python str
        fp = fp_bytes.numpy().decode("utf-8")
        img = preprocess_image(fp)          # returns float32 (H, W, 3)
        return img

    img = tf.py_function(
        func=_py_preprocess,
        inp=[filepath],
        Tout=tf.float32,
    )
    # Set the static shape so Keras knows the tensor dimensions
    # (py_function outputs have unknown shape by default)
    img.set_shape([Config.IMG_SIZE, Config.IMG_SIZE, 3])

    # One-hot encode the label for categorical cross-entropy
    label_oh = tf.one_hot(tf.cast(label, tf.int32), depth=Config.NUM_CLASSES)
    return img, label_oh


def build_tf_dataset(
    filepaths: List[str],
    labels: List[int],
    batch_size: int = Config.BATCH_SIZE,
    shuffle: bool = True,
    augment_fn=None,
) -> tf.data.Dataset:
    """
    Build a tf.data.Dataset from file paths and integer labels.

    Args:
        filepaths:  List of absolute image file paths.
        labels:     Corresponding list of integer ICDR labels (0-4).
        batch_size: Number of images per batch.
        shuffle:    Whether to shuffle the dataset each epoch.
                    Should be True for training, False for val/test.
        augment_fn: Optional callable (image, label) -> (image, label)
                    applied after preprocessing for training augmentation.
                    Set in Section 3 once augmentation is defined.

    Returns:
        Batched, prefetched tf.data.Dataset ready for model.fit().
    """
    # Build a dataset of (filepath, label) pairs
    dataset = tf.data.Dataset.from_tensor_slices((filepaths, labels))

    if shuffle:
        # Buffer size = full dataset ensures true random shuffling
        dataset = dataset.shuffle(
            buffer_size=len(filepaths),
            seed=Config.SEED,
            reshuffle_each_iteration=True,  # re-shuffle every epoch
        )

    # Preprocess each image in parallel; AUTOTUNE lets TF choose the
    # number of parallel workers based on available CPU cores
    dataset = dataset.map(
        _preprocess_tf,
        num_parallel_calls=tf.data.AUTOTUNE,
    )

    # Apply augmentation if provided (training only)
    if augment_fn is not None:
        dataset = dataset.map(augment_fn, num_parallel_calls=tf.data.AUTOTUNE)

    # Batch and prefetch — prefetch(AUTOTUNE) overlaps batch preparation
    # with GPU computation, eliminating the data-loading bottleneck
    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    return dataset


print("[Section 2] tf.data pipeline functions defined.")
print("  build_tf_dataset() will be called in Section 4 after the train/val/test split.")
print("  Augmentation will be wired in via the augment_fn argument in Section 3.")

In [ ]:
# ============================================================
# SECTION 2.7 — SECTION SUMMARY
# Print preprocessing pipeline confirmation so we can verify
# before moving on to augmentation.
# ============================================================

print("=" * 60)
print("  SECTION 2 COMPLETE — Preprocessing")
print("=" * 60)
print(f"  crop_image_from_gray()  : black border removal")
print(f"  ben_graham_enhance()    : local contrast normalisation")
print(f"  preprocess_image()      : full pipeline (load -> crop -> resize -> enhance -> normalise)")
print(f"  build_tf_dataset()      : tf.data pipeline with parallel map + prefetch")
print(f"")
print(f"  Target resolution : {Config.IMG_SIZE} x {Config.IMG_SIZE}")
print(f"  Output dtype      : float32, values in [0.0, 1.0]")
print(f"  Comparison images : saved to {Config.REPORTS_DIR}/preprocessing_class*.png")
print("=" * 60)
print("  Next -> Section 3: Data Augmentation & Class Balancing")

## Section 3: Data Augmentation & Class Balancing

**Goal:** Address two separate but related problems:

1. **Generalisation** — augmentation artificially expands the training distribution
   so the model does not memorise specific image artefacts or orientations.
2. **Class imbalance** — augmentation alone does NOT fix the loss function's
   bias toward the majority class. We also compute and apply `class_weight`
   via `sklearn.utils.class_weight.compute_class_weight`, which re-scales
   each sample's contribution to the loss proportionally to the inverse
   frequency of its class. This is the correct and principled fix.

Augmentation transforms used: rotation (±20°), horizontal/vertical flip,
zoom (±10%), brightness jitter, contrast jitter. All are applied **only to
training images** — validation and test images are never augmented.

In [ ]:
# ============================================================
# SECTION 3.1 — KERAS AUGMENTATION LAYER (graph-mode)
#
# WHY Keras augmentation layers instead of imgaug or albumentations:
#   - They run inside the tf.data pipeline on the GPU (not the CPU),
#     so augmentation does not bottleneck training.
#   - They are part of the Keras functional API and behave correctly
#     at inference time (turned off automatically when training=False).
#   - They are seeded via tf.random, consistent with our global seed.
#
# Each transform is applied with a probability; not every image gets
# every transform on every pass — this stochasticity is intentional.
# ============================================================

def build_augmentation_layer() -> keras.Sequential:
    """
    Build a Keras Sequential augmentation pipeline.

    All layers only apply their transform during training
    (training=True); during validation/testing they act as identity
    functions. This is handled automatically by Keras.

    Augmentations applied:
    - RandomFlip horizontal + vertical: retinal images have no canonical
      orientation — a flipped fundus is a valid fundus.
    - RandomRotation ±20° (factor=0.055): slight rotations simulate
      different camera angles during fundus photography.
    - RandomZoom ±10%: simulates variability in image magnification
      across different cameras / patient eye sizes.
    - RandomBrightness ±15%: compensates for variability in
      ophthalmoscope illumination intensity between clinics.
    - RandomContrast ±15%: compensates for variability in image
      processing applied by different fundus cameras.

    Returns:
        A compiled keras.Sequential model (used as a callable layer).
    """
    augmentation = keras.Sequential(
        [
            # Horizontal flip — retinal images are symmetric along vertical axis
            layers.RandomFlip("horizontal", seed=Config.SEED),
            # Vertical flip — less common clinically but increases variety
            layers.RandomFlip("vertical", seed=Config.SEED),
            # Rotation: factor is fraction of 2*pi, so 0.055 ≈ ±20 degrees
            layers.RandomRotation(factor=0.055, seed=Config.SEED),
            # Zoom: tuple means (height_factor, width_factor); negative = zoom out
            layers.RandomZoom(height_factor=(-0.1, 0.1), seed=Config.SEED),
            # Brightness: factor of 0.15 means ±15% brightness shift
            layers.RandomBrightness(factor=0.15, seed=Config.SEED),
            # Contrast: factor of 0.15 means contrast is scaled by [1-0.15, 1+0.15]
            layers.RandomContrast(factor=0.15, seed=Config.SEED),
        ],
        name="augmentation_pipeline",
    )
    return augmentation


# Build the augmentation layer once and reuse it
augmentation_layer = build_augmentation_layer()
print("[Augmentation] Layer built:")
augmentation_layer.summary()

In [ ]:
# ============================================================
# SECTION 3.2 — AUGMENTATION WRAPPER FOR tf.data
# Wraps the Keras augmentation layer in a function signature
# compatible with tf.data.Dataset.map().
# ============================================================

def augment_fn(image: tf.Tensor, label: tf.Tensor) -> tuple:
    """
    Apply augmentation to a single (image, label) pair.

    Used as the augment_fn argument to build_tf_dataset() so augmentation
    runs inside the tf.data pipeline. The training=True flag activates
    the random transforms; during inference (training=False) no transform
    is applied — this is handled automatically by Keras random layers.

    Args:
        image: float32 tensor (IMG_SIZE, IMG_SIZE, 3), values in [0, 1].
        label: one-hot float32 tensor of shape (NUM_CLASSES,).

    Returns:
        Tuple (augmented_image, label) — label is unchanged.
    """
    # training=True ensures the random transforms are active
    # The augmentation layer clips output to [0, 1] automatically
    image = augmentation_layer(image, training=True)
    return image, label


print("[Augmentation] augment_fn() defined and ready for build_tf_dataset().")

In [ ]:
# ============================================================
# SECTION 3.3 — CLASS WEIGHT COMPUTATION
#
# WHY class weighting in addition to augmentation:
#
# Augmentation increases the number of training samples but does NOT
# change the ratio of classes. If No DR has 14,063 samples and
# Proliferative has 702, augmentation with the same factor applied
# to all classes still leaves No DR ~20x more frequent.
#
# The loss function's gradient updates are proportional to the number
# of samples per class seen during training. Without correction, the
# model learns to be biased toward No DR because optimising for
# majority-class accuracy is the path of least resistance for the
# optimiser.
#
# class_weight re-scales each sample's gradient contribution so that
# a Proliferative DR sample contributes ~20x more to the loss update
# than a No DR sample — balancing the effective learning signal.
#
# Formula (sklearn "balanced"):
#   w_i = n_samples / (n_classes * n_samples_in_class_i)
#
# This is equivalent to training on a perfectly balanced dataset
# without the cost of physically duplicating minority-class samples.
# ============================================================

def compute_class_weights(labels: List[int]) -> Dict[int, float]:
    """
    Compute per-class loss weights using the "balanced" strategy.

    Weights are inversely proportional to class frequency so that the
    total weighted gradient contribution of every class is equal,
    regardless of how many raw samples each class has.

    Args:
        labels: List of integer ICDR labels (0-4) for the training set.

    Returns:
        Dict mapping class integer -> float weight, e.g. {0: 0.36, 4: 7.14}.
    """
    unique_classes = sorted(set(labels))  # [0, 1, 2, 3, 4]

    # sklearn compute_class_weight returns an array aligned to unique_classes
    weights_array = compute_class_weight(
        class_weight="balanced",
        classes=np.array(unique_classes),
        y=np.array(labels),
    )

    class_weight_dict = {cls: float(w) for cls, w in zip(unique_classes, weights_array)}

    print("\n[Class Weights] Computed using sklearn balanced strategy:")
    print("-" * 45)
    for cls, w in class_weight_dict.items():
        bar = "#" * int(w * 3)  # simple ASCII bar proportional to weight
        print(f"  Class {cls} ({Config.CLASS_NAMES[cls]:20s}): {w:6.3f}  {bar}")
    print("-" * 45)
    print("  Higher weight = minority class = larger gradient contribution per sample.")
    print("  These weights are passed to model.fit(class_weight=...) in Section 6.\n")

    return class_weight_dict


# NOTE: class weights must be computed on TRAINING labels only.
# We compute them here from the full dataset as a preview, and recompute
# on the actual train split in Section 4 after the stratified split.
_all_labels = df_labels["label"].tolist()
class_weights_preview = compute_class_weights(_all_labels)
print("[Note] Final class weights will be recomputed from training-split labels in Section 4.")

In [ ]:
# ============================================================
# SECTION 3.4 — VISUALISE AUGMENTED SAMPLES
# Show examples of augmented images for the report.
# Helps verify that the augmentations look realistic (not so
# extreme that they destroy diagnostic features).
# ============================================================

def visualise_augmented_samples(
    df: pd.DataFrame,
    n_images: int = 4,
    n_augmentations: int = 4,
    save_path: str = None,
) -> None:
    """
    Display and save a grid of original and augmented images.

    For each of n_images randomly sampled images, shows the original
    and n_augmentations different augmented versions side by side.
    This lets us visually confirm that augmentations are sensible
    (retina still visible, features not destroyed).

    Args:
        df:             DataFrame with 'filepath' and 'label' columns.
        n_images:       Number of source images to show.
        n_augmentations: Number of augmented versions per source image.
        save_path:      If provided, save the figure as a PNG.
    """
    sample = df.sample(n_images, random_state=Config.SEED)
    n_cols = n_augmentations + 1  # original + N augmented
    n_rows = n_images

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 3))
    if n_rows == 1:
        axes = axes[np.newaxis, :]

    # Column header labels
    col_labels = ["Original"] + [f"Aug {i+1}" for i in range(n_augmentations)]
    for ax, lbl in zip(axes[0], col_labels):
        ax.set_title(lbl, fontsize=9, fontweight="bold")

    for row_idx, (_, row) in enumerate(sample.iterrows()):
        # Load and preprocess the original (no augmentation)
        img_arr = preprocess_image(row["filepath"])          # float32, [0,1]
        img_tensor = tf.constant(img_arr)[tf.newaxis, ...]  # add batch dim

        # Column 0: original preprocessed image
        axes[row_idx, 0].imshow(img_arr)
        axes[row_idx, 0].axis("off")
        class_name = Config.CLASS_NAMES[row["label"]]
        axes[row_idx, 0].set_ylabel(
            f"Class {row['label']}: {class_name}", fontsize=7, rotation=90, labelpad=30
        )

        # Columns 1..n_augmentations: different augmented versions
        for aug_idx in range(n_augmentations):
            # Apply the augmentation layer (training=True activates random ops)
            aug_tensor = augmentation_layer(img_tensor, training=True)
            aug_arr = aug_tensor[0].numpy()  # remove batch dim
            # Clip to [0,1] in case brightness/contrast pushed values out of range
            aug_arr = np.clip(aug_arr, 0.0, 1.0)
            axes[row_idx, aug_idx + 1].imshow(aug_arr)
            axes[row_idx, aug_idx + 1].axis("off")

    fig.suptitle(
        "Original vs Augmented Fundus Images",
        fontsize=13, fontweight="bold", y=1.01,
    )
    plt.tight_layout()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"[Augmentation] Grid saved -> {save_path}")

    plt.show()


visualise_augmented_samples(
    df_labels,
    n_images=4,
    n_augmentations=4,
    save_path=os.path.join(Config.REPORTS_DIR, "augmentation_examples.png"),
)

In [ ]:
# ============================================================
# SECTION 3.5 — OVERFITTING MITIGATION SUMMARY
#
# This is a design-level comment block documenting all the anti-
# overfitting measures in use, for the report and code reader.
#
# Sources of overfitting protection in this pipeline:
#
# 1. DATA AUGMENTATION (this section)
#    Artificially increases training variety by applying random
#    geometric and photometric transforms. The model sees a different
#    version of each image every epoch, preventing memorisation of
#    exact pixel patterns.
#
# 2. CLASS WEIGHTING (this section)
#    Prevents the model from overfitting to the majority class (No DR)
#    by up-weighting minority-class gradient contributions.
#    Without this, "predict No DR always" achieves ~67% accuracy.
#
# 3. DROPOUT (Section 5)
#    Randomly zeroes 30% of neurons in the classification head during
#    training. Prevents co-adaptation of neurons and forces the model
#    to learn redundant representations. Removed at inference.
#
# 4. TWO-PHASE TRAINING WITH FROZEN BASE (Section 6)
#    Phase 1 keeps the pretrained EfficientNetB3 weights frozen so only
#    the lightweight head is trained. This prevents the pretrained
#    features from being immediately corrupted by noisy gradients from
#    a randomly-initialised head. Phase 2 uses a very low learning rate
#    (1e-5) to gently fine-tune without overwriting ImageNet features.
#
# 5. EARLY STOPPING (Section 6)
#    Monitors val_loss and stops training when it stops improving
#    (patience=5). Saves the best checkpoint, not the last epoch.
#
# 6. REDUCERLRONPLATEAU (Section 6)
#    Shrinks the learning rate when training stalls, preventing the
#    model from overshooting the loss minimum and oscillating.
# ============================================================

print("[Section 3] Overfitting mitigation measures documented above.")
print("  Augmentation  : rotation, flip, zoom, brightness, contrast")
print("  Class weight  : sklearn balanced weighting (recomputed on train split in Sec 4)")
print("  Dropout       : 0.3 — defined in classification head (Section 5)")
print("  Two-phase LR  : Phase 1 = 1e-3 (frozen), Phase 2 = 1e-5 (fine-tune)")
print("  Early Stopping: patience=5 on val_loss (Section 6)")
print("  LR Reduction  : ReduceLROnPlateau factor=0.5, patience=3 (Section 6)")

print("\n" + "=" * 60)
print("  SECTION 3 COMPLETE — Augmentation & Class Balancing")
print("=" * 60)
print("  augmentation_layer  : Keras Sequential (6 random transforms)")
print("  augment_fn()        : tf.data-compatible wrapper")
print("  compute_class_weights() : balanced inverse-frequency weighting")
print("  augmentation_examples.png saved to REPORTS_DIR")
print("  Next -> Section 4: Train / Validation / Test Split")
print("=" * 60)

## Section 4: Train / Validation / Test Split

**Goal:** Divide the dataset into three non-overlapping subsets and build
the final `tf.data` pipelines for each.

- **70% Training** — seen by the model during weight updates.
- **15% Validation** — seen during training for callback decisions
  (early stopping, LR reduction, checkpoint saving), but weights are
  never updated based on it.
- **15% Test** — held out completely until final evaluation in Section 7.

**Stratified splitting** (via `sklearn.model_selection.train_test_split`
with `stratify=`) ensures every split has approximately the same class
proportions as the full dataset. This is essential with an imbalanced
dataset: a random split could accidentally put all Proliferative DR
images into training, leaving the test set unable to evaluate that class.

After splitting, class weights are recomputed on training labels only
(not the full dataset) to prevent data leakage.

In [ ]:
# ============================================================
# SECTION 4.1 — STRATIFIED SPLIT
#
# WHY stratify=:
#   With 5 classes and severe imbalance (Proliferative DR ~702 images),
#   a simple random shuffle could place most Proliferative images in one
#   split by chance. stratify= guarantees each split mirrors the full
#   dataset class distribution proportionally.
#
# TWO-STEP SPLIT:
#   sklearn only does a single binary split (train vs rest).
#   To get three splits we split twice:
#     Step 1: full data -> 70% train | 30% temp
#     Step 2: temp      -> 50% val   | 50% test   (50% of 30% = 15% each)
# ============================================================

def stratified_split(
    df: pd.DataFrame,
    train_ratio: float = Config.TRAIN_RATIO,
    val_ratio: float   = Config.VAL_RATIO,
    test_ratio: float  = Config.TEST_RATIO,
    seed: int          = Config.SEED,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Split a labelled DataFrame into stratified train / val / test subsets.

    Uses two sequential calls to train_test_split with stratify= to ensure
    class proportions are preserved in all three subsets.

    Args:
        df:          Full labelled DataFrame with 'filepath' and 'label' columns.
        train_ratio: Fraction of data for training (default 0.70).
        val_ratio:   Fraction for validation (default 0.15).
        test_ratio:  Fraction for test (default 0.15).
        seed:        Random seed for reproducibility.

    Returns:
        Tuple (train_df, val_df, test_df) — three non-overlapping DataFrames.

    Raises:
        ValueError: If ratios do not sum to approximately 1.0.
    """
    total = train_ratio + val_ratio + test_ratio
    if abs(total - 1.0) > 1e-6:
        raise ValueError(
            f"Split ratios must sum to 1.0, got {total:.4f}. "
            f"Check Config.TRAIN_RATIO, VAL_RATIO, TEST_RATIO."
        )

    # Step 1: Split off the training set (70%)
    train_df, temp_df = train_test_split(
        df,
        test_size=(val_ratio + test_ratio),   # 30% goes to temp
        stratify=df["label"],                 # preserve class proportions
        random_state=seed,
    )

    # Step 2: Split the remaining 30% into val (15%) and test (15%)
    # The test_size for this split is test_ratio / (val_ratio + test_ratio)
    # = 0.15 / 0.30 = 0.50, so val and test are equal halves of temp
    val_df, test_df = train_test_split(
        temp_df,
        test_size=(test_ratio / (val_ratio + test_ratio)),
        stratify=temp_df["label"],
        random_state=seed,
    )

    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
    )


# Perform the split
train_df, val_df, test_df = stratified_split(df_labels)

print(f"[Split] Dataset divided:")
print(f"  Train : {len(train_df):>6,} images  ({len(train_df)/len(df_labels)*100:.1f}%)")
print(f"  Val   : {len(val_df):>6,} images  ({len(val_df)/len(df_labels)*100:.1f}%)")
print(f"  Test  : {len(test_df):>6,} images  ({len(test_df)/len(df_labels)*100:.1f}%)")
print(f"  Total : {len(train_df)+len(val_df)+len(test_df):>6,} images")

In [ ]:
# ============================================================
# SECTION 4.2 — PER-CLASS SIZE CONFIRMATION
# Print class counts for each split to confirm stratification
# worked correctly. All three splits should have roughly the
# same percentage breakdown as the full dataset.
# ============================================================

def print_split_distribution(
    train_df: pd.DataFrame,
    val_df:   pd.DataFrame,
    test_df:  pd.DataFrame,
) -> None:
    """
    Print per-class counts for all three splits side by side.

    Allows visual verification that stratification preserved class
    proportions. Large deviations from expected ratios would indicate
    a problem with the stratified split.

    Args:
        train_df: Training split DataFrame.
        val_df:   Validation split DataFrame.
        test_df:  Test split DataFrame.
    """
    all_classes = sorted(train_df["label"].unique())

    header = f"{'Class':<5} {'Name':<22} {'Train':>7} {'Val':>7} {'Test':>7}  {'Train%':>7} {'Val%':>7} {'Test%':>7}"
    print("\n" + "=" * len(header))
    print("  PER-CLASS SPLIT SIZES")
    print("=" * len(header))
    print(header)
    print("-" * len(header))

    for cls in all_classes:
        n_train = (train_df["label"] == cls).sum()
        n_val   = (val_df["label"]   == cls).sum()
        n_test  = (test_df["label"]  == cls).sum()
        n_total = n_train + n_val + n_test

        pct_train = n_train / n_total * 100 if n_total else 0
        pct_val   = n_val   / n_total * 100 if n_total else 0
        pct_test  = n_test  / n_total * 100 if n_total else 0

        print(
            f"  {cls:<5} {Config.CLASS_NAMES[cls]:<22} "
            f"{n_train:>7,} {n_val:>7,} {n_test:>7,}  "
            f"{pct_train:>6.1f}% {pct_val:>6.1f}% {pct_test:>6.1f}%"
        )

    print("=" * len(header))
    print("  Expected: ~70% / ~15% / ~15% per class (stratification check)")
    print("=" * len(header) + "\n")


print_split_distribution(train_df, val_df, test_df)

In [ ]:
# ============================================================
# SECTION 4.3 — RECOMPUTE CLASS WEIGHTS ON TRAINING SET ONLY
#
# WHY recompute here (not use the full-dataset preview from Section 3):
#   class_weight must reflect only the training distribution.
#   Using weights computed on the full dataset would incorporate
#   val/test label counts into the training objective — a subtle
#   form of data leakage. The difference is small (70% vs 100% of
#   data) but it is the principled approach and correct by convention.
# ============================================================

# Recompute class weights on training labels only
train_labels = train_df["label"].tolist()
class_weight_dict = compute_class_weights(train_labels)

print("[Class Weights] Final weights (computed on training split only):")
for cls, w in class_weight_dict.items():
    print(f"  {cls} {Config.CLASS_NAMES[cls]:<22}: {w:.4f}")
print("\n  These will be passed as class_weight= to model.fit() in Section 6.")

In [ ]:
# ============================================================
# SECTION 4.4 — BUILD tf.data DATASETS
# Wire together the preprocessing pipeline (Section 2),
# augmentation (Section 3), and the split DataFrames to produce
# three ready-to-use tf.data.Dataset objects.
#
# Key decisions:
#   - TRAINING dataset: shuffle=True, augment_fn applied
#   - VALIDATION dataset: shuffle=False, no augmentation
#     (we need consistent val metrics across epochs)
#   - TEST dataset: shuffle=False, no augmentation
#     (never seen during training; used once in Section 7)
# ============================================================

# Extract file paths and labels as plain Python lists
# (tf.data.from_tensor_slices accepts lists and NumPy arrays)
train_paths  = train_df["filepath"].tolist()
train_labels_list = train_df["label"].tolist()

val_paths    = val_df["filepath"].tolist()
val_labels_list   = val_df["label"].tolist()

test_paths   = test_df["filepath"].tolist()
test_labels_list  = test_df["label"].tolist()

# Build the three datasets
train_dataset = build_tf_dataset(
    filepaths=train_paths,
    labels=train_labels_list,
    batch_size=Config.BATCH_SIZE,
    shuffle=True,              # shuffle every epoch for training
    augment_fn=augment_fn,     # apply augmentation (training only)
)

val_dataset = build_tf_dataset(
    filepaths=val_paths,
    labels=val_labels_list,
    batch_size=Config.BATCH_SIZE,
    shuffle=False,             # fixed order for reproducible val metrics
    augment_fn=None,           # no augmentation at validation time
)

test_dataset = build_tf_dataset(
    filepaths=test_paths,
    labels=test_labels_list,
    batch_size=Config.BATCH_SIZE,
    shuffle=False,             # fixed order for final evaluation
    augment_fn=None,           # no augmentation at test time
)

# Calculate the number of batches per split (for model.fit steps_per_epoch)
steps_per_epoch       = len(train_paths)  // Config.BATCH_SIZE
validation_steps      = len(val_paths)    // Config.BATCH_SIZE
test_steps            = len(test_paths)   // Config.BATCH_SIZE

print("[Datasets] tf.data pipelines built:")
print(f"  train_dataset  : {len(train_paths):,} images  | {steps_per_epoch} batches/epoch")
print(f"  val_dataset    : {len(val_paths):,} images  | {validation_steps} batches/epoch")
print(f"  test_dataset   : {len(test_paths):,} images  | {test_steps} batches (eval)")
print(f"  Batch size     : {Config.BATCH_SIZE}")
print(f"  Augmentation   : train only")
print(f"  Shuffle        : train only")

In [ ]:
# ============================================================
# SECTION 4.5 — BATCH SHAPE VERIFICATION
# Peek at one batch to confirm shapes and dtypes are correct
# before we pass these datasets to the model.
# ============================================================

def verify_dataset_batch(dataset: tf.data.Dataset, name: str) -> None:
    """
    Inspect one batch from a tf.data.Dataset and print shape/dtype info.

    Args:
        dataset: A batched tf.data.Dataset (image, label) pairs.
        name:    Human-readable name for the dataset (e.g. "train").
    """
    for images, labels in dataset.take(1):
        print(f"[Batch Check] {name} dataset:")
        print(f"  images shape : {images.shape}   dtype: {images.dtype}")
        print(f"  labels shape : {labels.shape}   dtype: {labels.dtype}")
        print(f"  images range : [{images.numpy().min():.3f}, {images.numpy().max():.3f}]")
        print(f"  labels sample: {labels[0].numpy()}  (one-hot, sum={labels[0].numpy().sum():.0f})")


verify_dataset_batch(train_dataset, "train")
verify_dataset_batch(val_dataset,   "val")
verify_dataset_batch(test_dataset,  "test")

print("\n" + "=" * 60)
print("  SECTION 4 COMPLETE — Train / Val / Test Split")
print("=" * 60)
print(f"  train_df       : {len(train_df):,} rows")
print(f"  val_df         : {len(val_df):,} rows")
print(f"  test_df        : {len(test_df):,} rows")
print(f"  class_weight_dict : computed on training labels")
print(f"  train_dataset, val_dataset, test_dataset : ready for model.fit()")
print("  Next -> Section 5: Model — EfficientNetB3 Transfer Learning")
print("=" * 60)